In [ ]:
# NOTE: Installing transformers from the Git main branch is required because the \n# CohereAsrForConditionalGeneration model is very new and not included in the \n# stable release of transformers yet.\n# You MUST restart the kernel after running this cell.\n%pip install git+https://github.com/huggingface/transformers.git

# Cohere Transcribe Evaluation

This notebook evaluates the Cohere Transcribe model (cohere-transcribe-03-2026) on a dataset of audio segments.
It uses the direct batch inference approach with the shared evaluation runner.

In [ ]:
#@title Install packages

import builtins

# The Magic Hack: Create a dummy class and inject it into Python's builtins 
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass

builtins.PeftConfigLike = DummyPeftConfig

import os
import json
import sys
import torch
from transformers import pipeline
from google.cloud import storage

# Import common utils
from common.gcs_utils import download_jsonl_manifest, upload_inference_results
from common.inference_pipeline_runner import run_inference_pipeline
from common.audio_utils import preprocess_audio_for_model

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# --- Configuration ---
MODEL_NAME = "CohereLabs/cohere-transcribe-03-2026"
SELECTED_MODEL_KEY = "cohere_transcribe_03_2026"

GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"

BATCH_SIZE = 4
LIMIT = 10

In [ ]:
#@title Load the model and processor
import torch
from transformers import AutoProcessor, CohereAsrForConditionalGeneration

print(f"Loading Cohere model and processor: {MODEL_NAME}")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = CohereAsrForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16, # Keep bfloat16 to save memory
    trust_remote_code=True
)

In [ ]:
#@title Define helper functions for evaluation runner
import os
from transformers.audio_utils import load_audio
from common.prompts import COMMON_SYSTEM_PROMPT, COMMON_USER_INSTRUCTION

def prompt_formatter(entry, local_path):
    """Constructs the formatted chat prompt and returns it along with the audio path."""
    ASR_PROMPT = f"<|audio|> {COMMON_USER_INSTRUCTION}"
    chat = [{"role": "system", "content": COMMON_SYSTEM_PROMPT}, {"role": "user", "content": ASR_PROMPT}]
    cohere_chat_prompt = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    return (cohere_chat_prompt, local_path)

def cohere_inference(model, prompts):
    """Runs inference using the model and processor directly."""
    outputs = []
    for p in prompts:
        try:
            # Load audio at 16kHz as required by Cohere
            audio = load_audio(p[1], sampling_rate=16000)
            
            # Process with the formatted chat prompt and language="en"
            inputs = processor(audio, sampling_rate=16000, text=p[0], return_tensors="pt", language="en")
            inputs.to(model.device, dtype=model.dtype)
            
            # Generate tokens
            out = model.generate(**inputs, max_new_tokens=256)
            
            # Slice out prompt tokens if input_ids is present in inputs
            if "input_ids" in inputs:
                new_tokens = out[0, inputs["input_ids"].shape[-1]:]
            else:
                new_tokens = out[0]
            outputs.append(new_tokens)
            
        except Exception as e:
            logger.error(f"Failed during inference for {p[1]}: {e}")
            outputs.append("[ERROR]")
                
    return outputs

def result_decoder(ans, model):
    """Extracts the transcription safely using the processor."""
    if isinstance(ans, str) and ans == "[ERROR]":
        return ""
    try:
        # ans is already the sliced token sequence, decode it directly
        return processor.decode(ans, skip_special_tokens=True)
    except Exception as e:
        logger.error(f"Decoding failed: {e}")
        return ""

In [ ]:
#@title Run Evaluation

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Run the generic batch evaluation
results_list = run_inference_pipeline(
    model=model, # Use the model directly (not pipeline)
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=cohere_inference,
    decode_fn=result_decoder,
    preprocess_fn=preprocess_audio_for_model,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT
)

In [ ]:

# Upload results directly to GCS from memory
gcs_uri = upload_inference_results(
    storage_client, 
    GCS_BUCKET, 
    PROJECT_NAME, 
    SELECTED_MODEL_KEY, 
    EXPERIMENT_NAME, 
    results_list
)